In [1]:
"""
Detección de caras + emociones en tiempo real con webcam
Dependencias:
    pip install ultralytics opencv-python deepface tf-keras
"""
import sys
from ultralytics import YOLO
from deepface import DeepFace
import cv2
import numpy as np

# ── Configuración ────────────────────────────────────────────────────────────

FACE_MODEL   = 'arnabdhar/YOLOv8-Face-Detection'
EMOTION_SKIP = 3          # analizar emoción cada N frames (rendimiento)
CONF_THRESH  = 0.5        # confianza mínima para detectar cara

# Paleta de colores por emoción (BGR)
EMOTION_COLORS = {
    'happy':     (0,   220, 90),
    'sad':       (200, 80,  40),
    'angry':     (30,  30,  220),
    'surprise':  (0,   200, 255),
    'fear':      (160, 30,  160),
    'disgust':   (30,  160, 80),
    'neutral':   (160, 160, 160),
}
DEFAULT_COLOR = (200, 200, 200)

EMOJIS = {
    'happy':    '😄',
    'sad':      '😢',
    'angry':    '😠',
    'surprise': '😲',
    'fear':     '😨',
    'disgust':  '🤢',
    'neutral':  '😐',
}

# ── Helpers ──────────────────────────────────────────────────────────────────

def draw_rounded_rect(img, pt1, pt2, color, thickness=2, r=12):
    """Dibuja un rectángulo con esquinas redondeadas."""
    x1, y1 = pt1
    x2, y2 = pt2
    cv2.line(img,  (x1+r, y1), (x2-r, y1), color, thickness)
    cv2.line(img,  (x1+r, y2), (x2-r, y2), color, thickness)
    cv2.line(img,  (x1, y1+r), (x1, y2-r), color, thickness)
    cv2.line(img,  (x2, y1+r), (x2, y2-r), color, thickness)
    cv2.ellipse(img, (x1+r, y1+r), (r, r), 180,  0, 90,  color, thickness)
    cv2.ellipse(img, (x2-r, y1+r), (r, r), 270,  0, 90,  color, thickness)
    cv2.ellipse(img, (x1+r, y2-r), (r, r),  90,  0, 90,  color, thickness)
    cv2.ellipse(img, (x2-r, y2-r), (r, r),   0,  0, 90,  color, thickness)


def draw_label(img, text, x, y, color, bg_alpha=0.6):
    """Etiqueta con fondo semitransparente."""
    font       = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.65
    thickness  = 2
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    pad = 6
    overlay = img.copy()
    cv2.rectangle(overlay,
                  (x - pad, y - th - pad),
                  (x + tw + pad, y + baseline + pad),
                  (20, 20, 20), -1)
    cv2.addWeighted(overlay, bg_alpha, img, 1 - bg_alpha, 0, img)
    cv2.putText(img, text, (x, y), font, font_scale, color, thickness, cv2.LINE_AA)


def draw_hud(frame, face_count, fps):
    """HUD de información en la esquina superior izquierda."""
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (260, 70), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)
    cv2.putText(frame, f'Caras detectadas: {face_count}',
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (220, 220, 220), 1, cv2.LINE_AA)
    cv2.putText(frame, f'FPS: {fps:.1f}',
                (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (100, 220, 100), 1, cv2.LINE_AA)
    # Instrucción salida
    cv2.putText(frame, 'Presiona Q para salir',
                (w - 210, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (160, 160, 160), 1, cv2.LINE_AA)


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
    print("[INFO] Cargando modelo de detección de caras...")
    face_model = YOLO("yolov8n.pt")  # se descarga automáticamente
    print("[INFO] Iniciando webcam...")
    cap = cv2.VideoCapture(1)
    if not cap.isOpened():
        print("[ERROR] No se pudo abrir la webcam.")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    frame_count    = 0
    emotion_cache  = {}   # {cara_id: (emocion, score)} — caché por posición
    fps            = 0.0
    timer          = cv2.getTickCount()

    print("[INFO] Ejecutando detección. Presiona 'Q' para salir.\n")

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1

        # ── FPS ──────────────────────────────────────────────────────────────
        tick   = cv2.getTickCount()
        fps    = cv2.getTickFrequency() / (tick - timer)
        timer  = tick

        # ── Detección de caras ───────────────────────────────────────────────
        results    = face_model(frame, conf=CONF_THRESH, verbose=False)
        boxes      = results[0].boxes
        face_count = len(boxes)

        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf            = float(box.conf[0])

            # Recorte de la cara para análisis de emoción
            face_crop = frame[max(0,y1):y2, max(0,x1):x2]

            # ── Emoción (cada EMOTION_SKIP frames o primera vez) ─────────────
            emotion_label = 'neutral'
            emotion_score = 0.0
            cache_key     = idx   # índice de cara en el frame actual

            if face_crop.size > 0 and (frame_count % EMOTION_SKIP == 0
                                       or cache_key not in emotion_cache):
                try:
                    analysis = DeepFace.analyze(
                        face_crop,
                        actions=['emotion'],
                        enforce_detection=False,
                        silent=True
                    )
                    if isinstance(analysis, list):
                        analysis = analysis[0]
                    emotion_label = analysis['dominant_emotion']
                    emotion_score = analysis['emotion'][emotion_label]
                    emotion_cache[cache_key] = (emotion_label, emotion_score)
                except Exception:
                    pass  # Si falla el análisis, mantener caché anterior

            if cache_key in emotion_cache:
                emotion_label, emotion_score = emotion_cache[cache_key]

            # ── Dibujo ───────────────────────────────────────────────────────
            color = EMOTION_COLORS.get(emotion_label.lower(), DEFAULT_COLOR)

            draw_rounded_rect(frame, (x1, y1), (x2, y2), color, thickness=2)

            # Etiqueta: emoción + confianza detección cara
            label = f"{emotion_label.upper()}  {emotion_score:.0f}%"
            draw_label(frame, label, x1, y1 - 8, color)

            # Mini barra de confianza de la cara
            bar_w = x2 - x1
            bar_h = 5
            filled = int(bar_w * conf)
            cv2.rectangle(frame, (x1, y2 + 4), (x2, y2 + 4 + bar_h), (40, 40, 40), -1)
            cv2.rectangle(frame, (x1, y2 + 4), (x1 + filled, y2 + 4 + bar_h), color, -1)

        # ── HUD ──────────────────────────────────────────────────────────────
        draw_hud(frame, face_count, fps)

        cv2.imshow("Detección de Caras + Emociones  |  YOLOv8 + DeepFace", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("[INFO] Saliendo...")
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == '__main__':
    main()


[INFO] Cargando modelo de detección de caras...
[INFO] Iniciando webcam...
[ERROR] No se pudo abrir la webcam.


[ WARN:0@3.889] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[video4linux2,v4l2 @ 0x1e68dd00] Cannot open video device /dev/video1: Permission denied
[video4linux2,v4l2 @ 0x1e68dd00] Cannot open video device /dev/video0: Permission denied
[ WARN:0@3.889] global cap.cpp:438 open VIDEOIO(FFMPEG): raised OpenCV exception:

OpenCV(4.13.0) /io/opencv/modules/videoio/src/cap_ffmpeg_impl.hpp:1220: error: (-2:Unspecified error) in function 'bool CvCapture_FFMPEG::open(const char*, int, const cv::Ptr<cv::IStreamReader>&, const cv::VideoCaptureParameters&)'
> VIDEOIO/FFMPEG: Camera index out of range (expected: 'index < device_list->nb_devices'), where
>     'index' is 1
> must be less than
>     'device_list->nb_devices' is 0


[ WARN:0@3.889] global obsensor_stream_channel_v4l2.cpp:82 xioctl ioctl: fd=-1, req=-2140645888
[ WARN:0@3.889] global obsensor_stream_channel_v4l2.cpp:138 queryUvcDeviceInfoList ioctl error return: 9
[ WARN:0@3.889] global obsensor_s

In [20]:
import cv2

def listar_camaras():
    index = 0
    arr = []
    while index < 5: # Prueba los primeros 5 índices
        cap = cv2.VideoCapture(index)
        if cap.read()[0]:
            arr.append(index)
            cap.release()
        index += 1
    return arr

# Úsalo así:
print(f"Cámaras disponibles: {listar_camaras()}")

Cámaras disponibles: []


[ WARN:0@1001.306] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@1001.306] global obsensor_stream_channel_v4l2.cpp:82 xioctl ioctl: fd=-1, req=-2140645888
[ WARN:0@1001.306] global obsensor_stream_channel_v4l2.cpp:138 queryUvcDeviceInfoList ioctl error return: 9
[ WARN:0@1001.306] global obsensor_stream_channel_v4l2.cpp:82 xioctl ioctl: fd=-1, req=-2140645888
[ WARN:0@1001.306] global obsensor_stream_channel_v4l2.cpp:138 queryUvcDeviceInfoList ioctl error return: 9
[ERROR:0@1001.306] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range
[ WARN:0@1001.306] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video1): can't open camera by index
[ WARN:0@1001.306] global obsensor_stream_channel_v4l2.cpp:82 xioctl ioctl: fd=-1, req=-2140645888
[ WARN:0@1001.306] global obsensor_stream_channel_v4l2.cpp:138 queryUvcDeviceInfoList ioctl error return: 9
[ WARN:0@1001.306] global obsensor_stream_channel_v4l2.cpp:82 xioct

In [21]:

import cv2
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
print('¿Cámara abierta?', cap.isOpened())
ret, frame = cap.read()
if ret:
    cv2.imwrite('/tmp/test_camara.jpg', frame)
    print('✅ Foto guardada')
else:
    print('❌ Falla con CAP_V4L2')
cap.release()


¿Cámara abierta? False
❌ Falla con CAP_V4L2


[ WARN:0@1001.397] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ WARN:0@1001.398] global cap.cpp:478 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by index
